# Standoff Annotation and CoNLL-U

**Computational Text Analysis Pack · Module 20 — Encoding as Method**

*Technique:* Character-offset annotation; overlapping hierarchies; interchange formats

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/09-encoding-as-method/20c_standoff_and_conllu.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Inline XML has a hard structural limit: elements must nest. A sentence that runs
across a page break, or a quotation that starts inside one speech and ends
inside another, cannot be expressed — the tags would cross.

This is the **overlapping hierarchies** problem, and it is not a technicality.
Verse lines cross sentence boundaries; manuscript hands cross paragraph
boundaries; a scholar's thematic annotation crosses everything.

**Standoff annotation** solves it by storing annotations *outside* the text, as
character offsets. Nothing nests, so nothing can conflict.

## The tooling

TEI provides `<standOff>` for this; **CATMA** is built on standoff markup and is
widely used for literary annotation; the **STAM** model formalises it. For
linguistic annotation the interchange formats are **CoNLL-U** (Universal
Dependencies), **FoLiA**, and the **W3C Web Annotation** model. Annotation tools
— **INCEpTION**, **brat**, **Label Studio** — export to these.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
text = ("The Governor spoke at length. He said the treaty was signed in "
        "1885, and that the chiefs of Bonny had agreed to its terms.")

# Standoff: (start, end, layer, value). Nothing nests; layers are independent.
annotations = [
    (4, 12, "entity", "PERSON"),
    (0, 29, "sentence", "S1"),
    (30, 122, "sentence", "S2"),
    (43, 122, "quote", "reported-speech"),
    (66, 70, "date", "1885"),
    (93, 98, "entity", "PLACE"),
    (30, 70, "clause", "main"),
    (56, 122, "theme", "treaty-making"),
]
print(f"{'span':<12}{'layer':<12}{'value':<18}text")
print("-" * 74)
for s, e, layer, val in sorted(annotations):
    print(f"{f'{s}-{e}':<12}{layer:<12}{val:<18}{text[s:e][:34]!r}")

## Demonstrating the overlap

The `quote` span starts inside sentence S2 and the `theme` span starts inside
the quote and ends after it. In inline XML at least one of these has to be
broken into fragments or dropped. In standoff they simply coexist.

In [ ]:
def overlaps(a, b):
    return a[0] < b[1] and b[0] < a[1] and not (
        a[0] <= b[0] and b[1] <= a[1]) and not (
        b[0] <= a[0] and a[1] <= b[1])

crossing = [(a, b) for i, a in enumerate(annotations)
            for b in annotations[i + 1:] if overlaps(a, b)]
print(f"{len(crossing)} pairs of annotations CROSS (neither contains the other):\n")
for a, b in crossing:
    print(f"   {a[2]}:{a[3]} [{a[0]}-{a[1]}]  x  {b[2]}:{b[3]} [{b[0]}-{b[1]}]")
    print(f"      {text[min(a[0],b[0]):max(a[1],b[1])][:66]!r}")
print()
print("Each of these is an XML well-formedness error waiting to happen. As")
print("offsets they are simply four numbers that happen to interleave.")

In [ ]:
# Querying standoff: layers combine freely because they are independent.
def spans_in(layer):
    return [a for a in annotations if a[2] == layer]

print("Entities inside reported speech:")
for q in spans_in("quote"):
    for ent in spans_in("entity") + spans_in("date"):
        if q[0] <= ent[0] and ent[1] <= q[1]:
            print(f"   {ent[3]:<10}{text[ent[0]:ent[1]]!r}  inside {q[3]}")
print()
print("Themes that begin inside a quotation:")
for t in spans_in("theme"):
    for q in spans_in("quote"):
        if q[0] <= t[0] < q[1]:
            print(f"   {t[3]!r} starts inside {q[3]!r}")

## CoNLL-U: the linguistic interchange format

Where TEI models the *document*, **CoNLL-U** models the *sentence*: one token
per line, ten tab-separated columns, with dependency relations by index. It is
the format Universal Dependencies uses, which makes it the closest thing to a
cross-linguistic standard for annotated text — and therefore the right format
when you want to compare across languages.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("The chiefs of Bonny had agreed to its terms.")

COLS = ["ID", "FORM", "LEMMA", "UPOS", "XPOS", "FEATS", "HEAD", "DEPREL",
        "DEPS", "MISC"]
print("# text = " + doc.text)
print("\t".join(COLS))
print("-" * 96)
for i, tok in enumerate(doc, 1):
    head = 0 if tok.head is tok else tok.head.i + 1
    row = [str(i), tok.text, tok.lemma_, tok.pos_, tok.tag_,
           str(tok.morph) or "_", str(head), tok.dep_, "_",
           "_" if tok.whitespace_ else "SpaceAfter=No"]
    print("\t".join(row))
print()
print("HEAD is an index into this same table, so the syntax tree travels with")
print("the tokens. That is why CoNLL-U survives being emailed, versioned in git")
print("and read by a tool that knows nothing about the language.")

In [ ]:
# Why the format matters: the same structure, queried.
print("Dependency structure as a query target\n")
for tok in doc:
    if tok.dep_ in ("nsubj", "dobj", "pobj", "prep"):
        print(f"   {tok.dep_:<8}{tok.text:<10}-> head {tok.head.text!r}")
print()
print("A dependency-annotated corpus lets you ask 'which nouns are the SUBJECT")
print("of verbs of speaking' — a grammatical question that neither regex nor")
print("our token-level CQL from 15b could express, because both lack the tree.")

### 🔧 Your turn 1

Add a `page` layer with breaks that fall mid-sentence, then find every sentence that crosses a page boundary.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

This is the canonical overlapping-hierarchy example from the history of text
encoding — the physical structure of the book (pages, quires, columns) and its
logical structure (sentences, paragraphs, chapters) genuinely do not nest, and
the field spent years on the problem before standoff became the accepted answer.
Finding the crossings is a two-line filter over offsets. Expressing the same
thing in inline TEI requires `<milestone/>` empty elements as a workaround,
which is exactly what TEI recommends and exactly why `<standOff>` was added.

</details>